In [1]:
from plotly.io import show
from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from skfolio import Population, RatioMeasure
from skfolio.datasets import load_ftse100_dataset
from skfolio.model_selection import (
    CombinatorialPurgedCV,
    cross_val_predict,
    optimal_folds_number,
)
from skfolio.optimization import MeanRisk, ObjectiveFunction
from skfolio.pre_selection import DropCorrelated, DropZeroVariance
from skfolio.preprocessing import prices_to_returns

prices = load_ftse100_dataset()

X = prices_to_returns(prices)
X_train, X_test = train_test_split(X, test_size=0.33, shuffle=False)

In [2]:
model1 = MeanRisk(objective_function=ObjectiveFunction.MAXIMIZE_RATIO)
model1.fit(X_train)
model1.weights_

array([5.72489249e-08, 6.92799374e-02, 2.99565070e-02, 5.15427369e-07,
       5.98248544e-08, 2.11954169e-07, 1.45472665e-07, 6.30531935e-08,
       1.38474971e-01, 5.39755883e-04, 1.19668650e-06, 2.04407314e-07,
       8.79495071e-03, 9.56173668e-08, 9.05345545e-08, 3.00727419e-07,
       8.30609346e-02, 3.48492427e-07, 1.18995205e-07, 1.41496349e-07,
       3.13544690e-02, 9.49328191e-08, 3.89233565e-02, 6.73529748e-08,
       1.08791263e-01, 1.03983577e-07, 1.81104294e-01, 1.71678032e-07,
       6.17026728e-08, 1.85876178e-07, 1.06482316e-07, 5.09757681e-08,
       1.97934915e-06, 4.57814907e-08, 8.41631508e-02, 6.52279086e-08,
       6.16740873e-03, 1.07868906e-07, 1.72699954e-07, 8.59571975e-08,
       1.12484994e-01, 1.77846765e-07, 7.68211300e-08, 8.46476490e-08,
       9.91391605e-08, 1.08820474e-07, 9.52131433e-08, 4.71020369e-07,
       9.27208521e-08, 1.40414870e-07, 3.82148954e-03, 8.28101461e-02,
       3.04499311e-03, 7.93434222e-08, 1.98631698e-07, 1.72168221e-02,
      

In [3]:
set_config(transform_output="pandas")

model2 = Pipeline(
    [
        ("drop_zero_variance", DropZeroVariance(threshold=1e-6)),
        ("drop_correlated", DropCorrelated(threshold=0.5)),
        ("optimization", MeanRisk(objective_function=ObjectiveFunction.MAXIMIZE_RATIO)),
    ]
)
model2.fit(X_train)
model2.named_steps["optimization"].weights_

array([8.18629046e-02, 2.99990921e-02, 1.16397541e-06, 3.33347825e-07,
       1.82548038e-01, 2.85482485e-03, 3.56265072e-06, 1.10513944e-02,
       2.32253197e-07, 2.05141268e-07, 7.34710302e-07, 8.21495217e-02,
       9.52097597e-07, 3.41747905e-07, 3.58028595e-02, 4.20420351e-02,
       1.56245104e-07, 2.35646825e-07, 1.85053358e-01, 3.80020780e-07,
       2.42398710e-07, 1.15032452e-03, 1.09539530e-07, 9.00503120e-02,
       2.41991003e-07, 3.83742727e-07, 1.23265556e-01, 3.91693342e-07,
       1.83251807e-07, 2.09337576e-07, 2.38594386e-07, 2.26123969e-07,
       1.11576161e-06, 2.18898633e-07, 8.33465412e-03, 8.57587244e-02,
       1.29547016e-02, 1.87531533e-07, 4.25937609e-07, 2.51052265e-02,
       2.73616225e-07, 3.03373275e-07, 5.56618079e-07, 3.50951178e-07,
       1.05787166e-06, 1.45723458e-06])

In [4]:
ptf1 = model1.predict(X_test)
ptf1.name = "model1"
ptf2 = model2.predict(X_test)
ptf2.name = "model2"

print(ptf1.n_assets)
print(ptf2.n_assets)

64
46


In [5]:
population = Population([ptf1, ptf2])

In [6]:
population.plot_cumulative_returns()

In [7]:
#Combinatorial Purged Cross-Validation
n_folds, n_test_folds = optimal_folds_number(
    n_observations=X_test.shape[0],
    target_n_test_paths=100,
    target_train_size=800,
)

cv = CombinatorialPurgedCV(n_folds=n_folds, n_test_folds=n_test_folds)
cv.summary(X_test)

Number of Observations             1967
Total Number of Folds                10
Number of Test Folds                  6
Purge Size                            0
Embargo Size                          0
Average Training Size               786
Number of Test Paths                126
Number of Training Combinations     210
dtype: int64

In [8]:
pred_1 = cross_val_predict(
    model1,
    X_test,
    cv=cv,
    n_jobs=-1,
    portfolio_params=dict(annualized_factor=252, tag="model1"),
)

pred_2 = cross_val_predict(
    model2,
    X_test,
    cv=cv,
    n_jobs=-1,
    portfolio_params=dict(annualized_factor=252, tag="model2"),
)

In [9]:
population = pred_1 + pred_2

In [10]:
fig = population.plot_distribution(
    measure_list=[RatioMeasure.SHARPE_RATIO], tag_list=["model1", "model2"], n_bins=40
)
show(fig)

In [11]:
print(
    "Average of Sharpe Ratio:"
    f" {pred_1.measures_mean(measure=RatioMeasure.ANNUALIZED_SHARPE_RATIO):0.2f}"
)
print(
    "Std of Sharpe Ratio:"
    f" {pred_1.measures_std(measure=RatioMeasure.ANNUALIZED_SHARPE_RATIO):0.2f}"
)

Average of Sharpe Ratio: 0.46
Std of Sharpe Ratio: 0.20


In [12]:
print(
    "Average of Sharpe Ratio:"
    f" {pred_2.measures_mean(measure=RatioMeasure.ANNUALIZED_SHARPE_RATIO):0.2f}"
)
print(
    "Std of Sharpe Ratio:"
    f" {pred_2.measures_std(measure=RatioMeasure.ANNUALIZED_SHARPE_RATIO):0.2f}"
)

Average of Sharpe Ratio: 0.51
Std of Sharpe Ratio: 0.21
